In [ ]:
import mercury as mr 
from IPython.display import clear_output

In [ ]:
file = mr.UploadFile(label="Upload your Excel file", accept=".xlsx")

In [ ]:
import re
import unicodedata

def normalize_column_name(column_name):
    # Convert to string
    column_name = str(column_name)

    # Remove accents
    column_name = unicodedata.normalize("NFKD", column_name)
    column_name = column_name.encode("ascii", "ignore").decode("ascii")

    # Convert to lowercase
    column_name = column_name.lower()

    # Remove extra spaces
    column_name = column_name.strip()

    # Replace special characters with _
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)

    # Remove _ from start and end
    column_name = column_name.strip("_")

    return column_name

def normalize_column_names(df):
    df = df.copy()
    df.columns = [normalize_column_name(col) for col in df.columns]
    return df

def remove_duplicate_rows(df):
    # Make a copy
    df_clean = df.copy()

    # Remove duplicate rows
    df_clean = df_clean.drop_duplicates()

    return df_clean

In [ ]:
if file.name is not None:
    import pandas as pd
    from io import BytesIO
    data = BytesIO(file.value)
    excel = pd.ExcelFile(data)
    sheet_names = excel.sheet_names
    sheets_info = []
    df_list = []
    ready_df = []
    
    for sheet_name in excel.sheet_names:
        df = pd.read_excel(data, sheet_name=sheet_name)
        df_list.append(df)
        ready_df.append(df)
        rows = df.shape[0]
        columns = df.shape[1]
        empty_cells = df.isna().sum().sum()
        duplicate_rows = df.duplicated().sum()
    
        sheets_info.append({
            "Sheet Name": sheet_name,
            "Rows": rows,
            "Columns": columns,
            "Empty Cells": empty_cells,
            "Duplicate Rows": duplicate_rows
        })
    
    sheets_info_df = pd.DataFrame(sheets_info)

In [ ]:
if file.name is None:
    mr.Markdown("# Upload the Excel file to start") 
else:
    mr.Markdown(f"# Uploaded file: {file.name}", key='title_md')

In [ ]:
if file.name is not None:
    sheets_info_table = mr.Table(sheets_info_df, page_size=20, key='sheets-info')
else:
    clear_output()

In [ ]:
if file.name is not None: 
    sheet_select = mr.Select(label="Choose sheet", choices=sheet_names)
else: 
    clear_output()

In [ ]:
if file.name is not None: 
    mr.Markdown(f"## Sheet: {sheet_select.value}", key='sheet_md')
else:
    clear_output()

In [ ]:
if file.name is not None:
    oryginal_data_table = mr.Table(df_list[sheet_names.index(sheet_select.value)], page_size=20, key=f"oryginal-df-{sheet_select.value}")
else: 
    clear_output()

In [ ]:
if file.name is not None: 
    mr.Markdown("### Choose operation", position="sidebar", key='checkboxes')
    normalize_col_names_checkbox = mr.CheckBox(label="Normalize column names", appearance="box", key=f"normalize_col_names_checkbox-{sheet_select.value}")
    remove_duplicate_rows_checkbox = mr.CheckBox(label="Remove duplicate rows", appearance="box", key=f"remove_duplicate_rows_checkbox-{sheet_select.value}")
else: 
    clear_output()

In [ ]:
if file.name is not None:
    if normalize_col_names_checkbox.value:
        normalize_col_names_checkbox.disabled = True
        ready_df[sheet_names.index(sheet_select.value)] = normalize_column_names(ready_df[sheet_names.index(sheet_select.value)])

    if remove_duplicate_rows_checkbox.value:
        remove_duplicate_rows_checkbox.disabled = True
        ready_df[sheet_names.index(sheet_select.value)] = remove_duplicate_rows(ready_df[sheet_names.index(sheet_select.value)])

In [ ]:
if file.name is not None: 
    if normalize_col_names_checkbox.value or remove_duplicate_rows_checkbox.value: 
        mr.Markdown("## Results", key='results')
        edited_data_table = mr.Table(ready_df[sheet_names.index(sheet_select.value)], page_size=20, key=f"results-table-{sheet_select.value}")
    else:
        clear_output()

In [ ]:
if file.name is not None:
    csv_data = ready_df[sheet_names.index(sheet_select.value)].to_csv(index=False)

In [ ]:
if file.name is not None:
    if normalize_col_names_checkbox.value or remove_duplicate_rows_checkbox.value:
        mr.Markdown("### Download edited sheet", position="sidebar", key='download')
        mr.Download(
            data=csv_data,
            filename=f"{normalize_column_name(sheet_select.value)}-edited.csv",
            mime="text/csv",
            label="Download as CSV",
            key=f"csv-download-{sheet_select.value}-{normalize_col_names_checkbox.value}-{remove_duplicate_rows_checkbox.value}"
        )
    else: 
        clear_output()